Loading and filter to frontal views only

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv("../data/raw/chexpert/train.csv")
df["patient_id"] = df["Path"].str.extract(r"(patient\d+)")

df = df[df["Frontal/Lateral"] == "Frontal"].copy()
print("After frontal filter:", df.shape)

After frontal filter: (191027, 20)


Assigning a single primary label per image:

In [ ]:
pathology_cols = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", 
                   "Lung Opacity", "Lung Lesion", "Edema", "Consolidation", 
                   "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", 
                   "Pleural Other", "Fracture", "Support Devices"]

# Priority order: highlight rarer pathologies first, so they aren't masked by common ones
priority_order = ["Fracture", "Pneumonia", "Lung Lesion", "Pleural Other", 
                   "Pneumothorax", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
                   "Cardiomegaly", "Pleural Effusion", "Atelectasis", "Lung Opacity",
                   "Support Devices", "No Finding"]

def assign_primary_label(row):
    for col in priority_order:
        if row[col] == 1.0:
            return col, False  # not uncertain
        if row[col] == -1.0:
            return col, True   # uncertain, but still candidate
    return None, False

results = df[pathology_cols].apply(assign_primary_label, axis=1)
df["primary_label"] = results.apply(lambda x: x[0])
df["is_uncertain_original"] = results.apply(lambda x: x[1])

df = df.dropna(subset=["primary_label"])
print(df["primary_label"].value_counts())



df["clean_path"] = df["Path"].str.replace(
    "CheXpert-v1.0-small/", "", regex=False
)

primary_label
Edema                         39439
Consolidation                 27949
Pleural Effusion              22251
Pneumonia                     20116
Pneumothorax                  17087
No Finding                     9474
Support Devices                8924
Fracture                       7935
Cardiomegaly                   7871
Enlarged Cardiomediastinum     7829
Lung Lesion                    6567
Atelectasis                    5229
Lung Opacity                   4389
Pleural Other                  3207
Name: count, dtype: int64


Cap "No Finding", sample remaining classes:

In [ ]:
TARGET_TOTAL     = 10000
NO_FINDING_CAP   = 1500
UNCERTAIN_TARGET = 1000

no_finding_df = df[df["primary_label"] == "No Finding"].sample(
    n=min(NO_FINDING_CAP, (df["primary_label"] == "No Finding").sum()), random_state=42)

pathology_df = df[df["primary_label"] != "No Finding"]

# guaranteeing a portion of originally-uncertain samples for manifest
uncertain_pool = pathology_df[pathology_df["is_uncertain_original"] == True]
uncertain_sample = uncertain_pool.sample(
    n=min(UNCERTAIN_TARGET, len(uncertain_pool)), random_state=42)

remaining_budget = TARGET_TOTAL - len(no_finding_df) - len(uncertain_sample)

certain_pool = pathology_df[~pathology_df.index.isin(uncertain_sample.index)]
classes_remaining = certain_pool["primary_label"].unique()
per_class_target = remaining_budget // len(classes_remaining)

sampled_parts = []
for cls in classes_remaining:
    cls_df = certain_pool[certain_pool["primary_label"] == cls]
    n = min(per_class_target, len(cls_df))
    sampled_parts.append(cls_df.sample(n=n, random_state=42))

certain_sample = pd.concat(sampled_parts)

chexpert_subset = pd.concat([no_finding_df, uncertain_sample, certain_sample]).reset_index(drop=True)
print(chexpert_subset.shape)
print(chexpert_subset["primary_label"].value_counts())

(9988, 23)
primary_label
No Finding                    1500
Consolidation                  946
Pneumonia                      860
Edema                          659
Enlarged Cardiomediastinum     657
Cardiomegaly                   615
Pneumothorax                   615
Pleural Effusion               611
Pleural Other                  605
Lung Lesion                    592
Atelectasis                    591
Lung Opacity                   580
Fracture                       579
Support Devices                578
Name: count, dtype: int64


Deduplicate by patient ID (keep first occurrence per patient):

In [4]:
chexpert_subset = chexpert_subset.drop_duplicates(subset="patient_id", keep="first")
print("After patient dedup:", chexpert_subset.shape)

After patient dedup: (8776, 23)


Build the standardised manifest schema and save:

In [5]:
manifest = pd.DataFrame({
    "image_id": chexpert_subset["Path"].apply(lambda x: x.split("/")[-1]),
    "image_path": chexpert_subset["Path"].str.replace("CheXpert-v1.0-small/", "", regex=False),
    
    "dataset": "chexpert",
    "label": chexpert_subset["primary_label"],
    "is_uncertain_original": chexpert_subset["is_uncertain_original"],
    "patient_id": chexpert_subset["patient_id"],
})

manifest.to_csv("../data/subsets/chexpert_subset_manifest.csv", index=False)
print("Saved:", manifest.shape)

Saved: (8776, 6)
